# Loading Tabular Data

The fastest way to get music data into TimeToAlign! is through **tabular loaders**. If your data is in CSV or TSV format, you're just 3 lines of code away from analysis.

**What you'll learn:**
- Load music annotations from TSV/CSV files
- Access event counts, coordinate ranges, and metadata
- Work with different coordinate types (seconds, beats, fractions)
- Create timelines from loaded data
- Create custom loaders for non-standard formats
- Parse JSON columns into structs for nested field access
- Use `Field` and `ComputedField` for advanced coordinate mapping

**Time:** 15 minutes

## TL;DR

```python
from timetoalign.loader.tabular import Ms3Loader

loader = Ms3Loader()
loader.load("beethoven.notes.tsv")

df = loader.events.to_pandas()       # Get as DataFrame
timeline = loader.create_timeline()  # Create Timeline
```

## Setup

In [1]:
from pathlib import Path

# Specimen directories
SPECIMENS = Path(".").resolve().parents[1] / ".." / "dashboard" / "specimens"
BEETHOVEN = SPECIMENS / "beethoven_woo71"
THORESEN = SPECIMENS / "thoresen"

# Available files
{
    "Beethoven files": [f.name for f in BEETHOVEN.glob("WoO71.*.tsv")],
    "Thoresen files": [f.name for f in THORESEN.glob("*.tsv")],
}

{'Beethoven files': ['WoO71.chords.tsv',
  'WoO71.measures.tsv',
  'WoO71.notes.tsv'],
 'Thoresen files': ['thoresen_test_h.tsv', 'thoresen_test.tsv']}

## Loading Notes from TSV

The `Ms3Loader` handles TSV files exported from the [ms3](https://github.com/johentsch/ms3) parser, which processes MuseScore files.

**Three lines of code:**

In [2]:
from timetoalign.loader.tabular import Ms3Loader

loader = Ms3Loader()
loader.load(BEETHOVEN / "WoO71.notes.tsv")

f"{len(loader.events):,} notes loaded"

'4,753 notes loaded'

## Converting to pandas

Use `to_pandas()` to get a DataFrame with clean coordinate values:

In [3]:
df = loader.events.to_pandas()
df.head()

,id,name,temporal_type,event_type,start,end,duration,mc,staff,mn,chord_id,voice,midi,octave,tpc
0,e000000,A3,interval,Note,0,1/4,1/4,1,2,0,3,1,57,3,3
1,e000001,E4,interval,Note,0,1/4,1/4,1,1,0,2,2,64,4,4
2,e000002,A4,interval,Note,0,1/8,1/8,1,1,0,0,1,69,4,3
3,e000003,C#5,interval,Note,0,1/8,1/8,1,1,0,0,1,73,5,7
4,e000004,E5,interval,Note,1/2,5/8,1/8,1,1,0,1,1,76,5,4


In [4]:
# Coordinates are shown as Fractions when available
row = df.iloc[0]
{
    "start": row["start"],
    "start_type": type(row["start"]).__name__,
    "end": row["end"],
    "duration": row["duration"],
}

{'start': Fraction(0, 1),
 'start_type': 'Fraction',
 'end': Fraction(1, 4),
 'duration': Fraction(1, 4)}

## Quick Statistics

The loader provides immediate access to summary information:

In [5]:
{
    "event_count": len(loader.events),
    "coordinate_range": loader.events.coordinate_range(),
    "unit": str(loader.unit),
    "number_type": str(loader.number_type),
}

{'event_count': 4753,
 'coordinate_range': (0.0, 877.75),
 'unit': 'quarters',
 'number_type': 'fraction'}

In [6]:
# Event types (all notes in this file)
loader.count_events_by_type()

{'Note': 4753}

In [7]:
# Temporal types: interval (has duration) vs instant (no duration)
loader.count_events_by_temporal_type()

{'interval': 4753}

## Creating Timelines

TimeToAlign! represents temporal data as **Timelines**. There are three equivalent ways to create a timeline from loaded data:

In [8]:
# Method 1: From the loader directly (most convenient)
timeline = loader.create_timeline(uid="beethoven_notes")
{
    "timeline": str(timeline),
    "length": f"{timeline.length} {timeline.unit}",
}

{'timeline': 'ContinuousLogicalTimeline[beethoven_notes] (1 children)\n0 ____________________________________________________ 877.8 quarters\n  └─ events           0 ____________________________________________________ 877.8',
 'length': '877.75 quarters quarters'}

In [9]:
# Method 2: From the events directly
events_timeline = loader.events.create_timeline(uid="from_events")
events_timeline

ContinuousLogicalTimeline(id='from_events', length=877.75, unit=quarters, events=4753, children=0)

In [10]:
# Method 3: From the store (useful when loader has multiple data types)
store_timeline = loader.store.create_timeline(uid="from_store")
store_timeline

ContinuousLogicalTimeline(id='from_store', length=877.75, unit=quarters, events=0, children=1)

## Performance

TabularLoaders are **vectorized** - they use numpy/pandas operations instead of Python loops. This means loading 10,000+ events takes milliseconds:

In [11]:
import time

start = time.perf_counter()
loader = Ms3Loader()
loader.load(BEETHOVEN / "WoO71.notes.tsv")
elapsed = time.perf_counter() - start

{
    "events": len(loader.events),
    "time_ms": f"{elapsed*1000:.1f}",
    "events_per_sec": f"{len(loader.events)/elapsed:,.0f}",
}

{'events': 4753, 'time_ms': '50.6', 'events_per_sec': '94,017'}

## Custom Loaders for Non-Standard Formats

For files that don't match the ms3 format, create a custom loader by subclassing `TsvLoader` or `CsvLoader`.

Let's load the Thoresen annotations file which has a different column structure:

In [12]:
# Preview the file structure
import pandas as pd

preview = pd.read_csv(THORESEN / "thoresen_test.tsv", sep="\t", nrows=3)
preview

,event_id,alignment_group_id,start_time_sec,duration_sec,event_type,graphical_element_id,image_filename,rect_coords_json,text_content,text_anchor_xy_json,layer_order,description
0,annot_cue_001,NaN,0.0,5.0,rectangle,rect_a,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,NaN,NaN,NaN
1,annot_cue_002,NaN,1.5,4.0,rectangle,rect_b,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,NaN,NaN,NaN
2,annot_cue_003,NaN,3.5,2.0,rectangle,rect_c,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,NaN,NaN,NaN


In [13]:
list(preview.columns)

['event_id',
 'alignment_group_id',
 'start_time_sec',
 'duration_sec',
 'event_type',
 'graphical_element_id',
 'image_filename',
 'rect_coords_json',
 'text_content',
 'text_anchor_xy_json',
 'layer_order',
 'description']

### Creating a Custom Loader

Override class attributes to map columns:

In [14]:
from timetoalign.loader.tabular import TsvLoader
from timetoalign.core import TimeUnit, NumberType

class ThoresenLoader(TsvLoader):
    """Loader for Thoresen annotation TSV files."""
    
    # Column mapping
    id_column = "event_id"
    start_column = "start_time_sec"
    duration_column = "duration_sec"
    event_type_column = "event_type"
    name_column = "description"
    
    # Coordinates are in seconds as floats
    _default_unit = TimeUnit.seconds
    coordinate_type = NumberType.float
    
    # Extra columns to include (just list the names)
    extra_columns = [
        "image_filename",
        "rect_coords_json",
        "graphical_element_id",
    ]

# Load the file
thoresen = ThoresenLoader()
thoresen.load(THORESEN / "thoresen_test.tsv")

thoresen.events.to_pandas()

,id,name,temporal_type,event_type,start,end,duration,rect_coords_json,image_filename,graphical_element_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_a
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_b
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_c
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,"{""x"": 145, ""y"": 90, ""width"": 160, ""height"": 58}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_a2
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,"{""x"": 385, ""y"": 46, ""width"": 139, ""height"": 20}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_h2
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,"{""x"": 310, ""y"": 93, ""width"": 154, ""height"": 18}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_d3
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,"{""x"": 456, ""y"": 69, ""width"": 229, ""height"": 18}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_b3
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,"{""x"": 14, ""y"": 115, ""width"": 127, ""height"": 31}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_i4
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,"{""x"": 663, ""y"": 82, ""width"": 97, ""height"": 23}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_a4
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,"{""x"": 19, ""y"": 119, ""width"": 251, ""height"": 29}",thoresen_2010_form-building-patterns_p90-91_pa...,rect_i5


In [15]:
thoresen.create_timeline(uid="thoresen_physical")

ContinuousPhysicalTimeline(id='thoresen_physical', length=142.5, unit=seconds, events=0, children=1)

### Advanced: Using Struct Fields and Computed Columns

The Thoresen data has a `rect_coords_json` column containing pixel coordinates as JSON:
```json
{"x": 10, "y": 90, "width": 148, "height": 55}
```

TimeToAlign! can **parse this JSON into a struct column** and use struct fields directly as coordinates. This enables creating **two different loaders** from the same file:

1. **ThoresenLoader**: Uses `start_time_sec` (seconds) - what we just created
2. **ThoresenGraphicalLoader**: Uses `rect_coords.x` (pixels)

Key classes:
- `ExtraField(name, dict, source=...)`: Parse JSON into a struct column
- `Field(column, field)`: Access a field within a struct
- `ComputedField(name, formula=...)`: Compute derived columns

In [16]:
from timetoalign.loader import Field, ComputedField, ExtraField
from timetoalign.loader.tabular import TsvLoader
from timetoalign.core import TimeUnit, NumberType

class ThoresenGraphicalLoader(TsvLoader):
    """Loader for Thoresen TSV using PIXEL coordinates from JSON struct.
    
    This creates a graphical timeline where:
    - start = rect_coords.x (left edge of rectangle)
    - end = rect_coords.x + rect_coords.width (right edge)
    """
    
    # Parse JSON column into a struct
    extra_columns = [
        ExtraField("rect_coords", dict, source="rect_coords_json"),
    ]
    
    # Use struct fields for coordinates
    start_column = Field("rect_coords", "x")
    end_column = ComputedField("end", formula="rect_coords.x + rect_coords.width")
    
    # Pixel coordinates
    _default_unit = TimeUnit.pixels
    coordinate_type = NumberType.float
    default_event_type = "Rectangle"

# Load the same file with pixel coordinates
graphical_loader = ThoresenGraphicalLoader()
graphical_loader.load(THORESEN / "thoresen_test.tsv")

{
    "unit": str(graphical_loader.unit),
    "coordinate_range": graphical_loader.events.coordinate_range(),
}

{'unit': 'pixels', 'coordinate_range': (10.0, 760.0)}

In [17]:
# Compare the two loaders side-by-side
import pyarrow.compute as pc

physical_df = thoresen.events.to_pandas()[["id", "start", "end"]].head()
physical_df.columns = ["id", "start_sec", "end_sec"]

# Get graphical coordinates
graphical_table = graphical_loader.events.table
starts_px = pc.struct_field(graphical_table.column("start"), "value").to_pylist()[:5]
ends_px = pc.struct_field(graphical_table.column("end"), "value").to_pylist()[:5]

physical_df["start_px"] = starts_px
physical_df["end_px"] = ends_px

physical_df

,id,start_sec,end_sec,start_px,end_px
0,annot_cue_001,0.0,5.0,10.0,158.0
1,annot_cue_002,1.5,5.5,40.0,167.0
2,annot_cue_003,3.5,5.5,111.0,168.0
3,annot_cue_004,34.6,39.8,145.0,305.0
4,annot_cue_005,43.5,48.0,385.0,524.0


**Syntax options for struct field access:**

```python
# All of these are equivalent:
start_column = Field("rect_coords", "x")           # Field object
start_column = ("rect_coords", "x")                # Tuple shorthand

# For computed columns:
end_column = ComputedField("end", formula="rect_coords.x + rect_coords.width")

# Or with a callable:
def compute_end(table):
    x = pc.struct_field(table["rect_coords"], "x")
    w = pc.struct_field(table["rect_coords"], "width")
    return pc.add(x, w)

end_column = ComputedField("end", expr=compute_end)
```

## Summary

In this notebook, we learned:

1. **Loading tabular data** with `Ms3Loader` and custom loaders
2. **Creating timelines** with `.create_timeline()` from loader, events, or store
3. **Extra columns** are specified as a simple list: `extra_columns = ["col1", "col2"]`
4. **Struct columns**: Parse JSON into structs with `ExtraField(name, dict, source=...)`
5. **Field access**: Use `Field(column, field)` or `(column, field)` for struct fields
6. **Computed columns**: Use `ComputedField(name, formula=...)` for derived values

### Key API

```python
# Load
loader = Ms3Loader()
loader.load("file.tsv")

# Access
df = loader.events.to_pandas()
timeline = loader.create_timeline()

# Custom loader
class MyLoader(TsvLoader):
    start_column = "onset"
    duration_column = "dur"
    extra_columns = ["pitch", "velocity"]

# Advanced: Struct fields and computed columns
from timetoalign.loader import Field, ComputedField, ExtraField

class GraphicalLoader(TsvLoader):
    extra_columns = [
        ExtraField("rect", dict, source="rect_json"),  # Parse JSON
    ]
    start_column = Field("rect", "x")                  # Struct field
    end_column = ComputedField("end", formula="rect.x + rect.width")
```

> **Key Takeaway:** Tabular loaders provide a declarative way to map CSV/TSV columns to TimeToAlign! events. The same file can create different timelines (physical vs graphical) by specifying different coordinate columns.